#### llm powered chaatbot , converstaion + remember previous conversation
#### -> conversational rag : enbale chatbot experience over an external resource of data 
#### -> agent : build a chatbot that takes action 

In [44]:
import os 
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
groq_api_key

'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'

In [45]:
from langchain_groq import ChatGroq

model = ChatGroq(model="Gemma2-9b-It" , groq_api_key = groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000002392772F680>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002392772E840>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [46]:
from langchain_core.messages import HumanMessage , SystemMessage

model.invoke([
    HumanMessage(content = "hello my name is naman nimble and i am a data scientist")
])

AIMessage(content="Hello Namman Nimble! It's nice to meet you. \n\nThat's a great name and a fascinating profession.  What kind of data science work do you do? \n\nI'm always interested in learning more about how people use data to solve problems and make discoveries.\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 21, 'total_tokens': 84, 'completion_time': 0.114545455, 'prompt_time': 0.002138416, 'queue_time': 0.235056879, 'total_time': 0.116683871}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-9281d365-fc84-4657-9365-d5b551f747a7-0', usage_metadata={'input_tokens': 21, 'output_tokens': 63, 'total_tokens': 84})

In [47]:
from langchain_core.messages import AIMessage
model.invoke([
    HumanMessage(content="hello my name is naman nimble and i am a data scientist"),
    AIMessage(content = "Hello Naman Nimble, it's nice to meet you! I'm Gemma, a language model trained by Langchain. How can I help you today?"),
    HumanMessage(content = "I am looking for a job in data science"),
    AIMessage(content = "Great! I can help you with that. What kind of job are you looking for?")
])

AIMessage(content=' \n\nTo give you the best advice, tell me:\n\n* **What industry are you interested in?** (e.g., healthcare, finance, tech, retail)\n* **What type of role are you seeking?** (e.g., Data Analyst, Machine Learning Engineer, Data Scientist)\n* **What are your key skills and areas of expertise?** (e.g., Python, R, SQL, deep learning, NLP)\n* **What location are you targeting?** (e.g., specific city, remote) \n\nThe more information you provide, the better I can tailor my suggestions to your needs.  \n\nI can also help with:\n\n* **Finding relevant job postings**\n* **Crafting a strong resume and cover letter**\n* **Preparing for data science interviews**\n\n\nGood luck with your job search! \n\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 181, 'prompt_tokens': 92, 'total_tokens': 273, 'completion_time': 0.329090909, 'prompt_time': 0.005511739, 'queue_time': 0.23398985200000003, 'total_time': 0.334602648}, 'model_name': 'Gemma2-9b-It', '

# using message history 


In [48]:
### message history 
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id : str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(model , get_session_history)

In [49]:
config = {
    "configurable" : {"session_id":"chat1"}
}

In [50]:
with_message_history.invoke([
    HumanMessage(content="hello my name is naman nimble and i am a data scientist"),
    AIMessage(content = "Hello Naman Nimble, it's nice to meet you! I'm Gemma, a language model trained by Langchain. How can I help you today?"),
    HumanMessage(content = "I am looking for a job in data science"),
    AIMessage(content = "Great! I can help you with that. What kind of job are you looking for?")
],config=config)

AIMessage(content=' \n\nTo give you the best advice, tell me:\n\n* **What industry are you interested in?** (e.g., healthcare, finance, tech)\n* **What kind of role are you targeting?** (e.g., data analyst, machine learning engineer, data scientist)\n* **What are your areas of expertise?** (e.g., Python, SQL, specific machine learning algorithms)\n* **What kind of company culture are you looking for?** (e.g., startup, large corporation, remote)\n* **Where are you located?** (or are you open to remote work?)\n\n\nThe more information you give me, the better I can tailor my suggestions to your specific needs.\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 92, 'total_tokens': 245, 'completion_time': 0.278181818, 'prompt_time': 0.005344439, 'queue_time': 0.233670086, 'total_time': 0.283526257}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-70462d96-c89d-4019-a

In [51]:
with_message_history.invoke([
    HumanMessage(content="what is my name ?")
], config=config)

AIMessage(content='Your name is Naman Nimble. 😊  \n\nIs there anything else I can help you with regarding your job search?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 268, 'total_tokens': 296, 'completion_time': 0.050909091, 'prompt_time': 0.012385266, 'queue_time': 0.235266645, 'total_time': 0.063294357}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-f349c81d-28ee-4ab6-b6d6-420a1fcf9a6e-0', usage_metadata={'input_tokens': 268, 'output_tokens': 28, 'total_tokens': 296})

# so it remembers the previous conversation and uses it to generate the next response.

In [52]:
## chaange the config --> session id 

config1 = {
    "configurable" : {"session_id":"chat2"}
}

response = with_message_history.invoke([
    HumanMessage(content="what is my name ?"
        )], config=config1)

response.content


"As an AI, I have no memory of past conversations and do not know your name. If you'd like to tell me, I'd be happy to use it! 😊  \n\n"

#  changin config -> session id will lead to historical memory loss

In [53]:
from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system" , " you are a helpful assistant , answer all the questions to the best of your abilities"),
        MessagesPlaceholder(variable_name="messages"), # key values pairs messages where messages is the key 
    ]
)

chain = prompt | model 

In [54]:
chain.invoke({
    "messages" : [HumanMessage(content="hello my name is Naman and i am a data scientist")]
})

AIMessage(content="Hello Naman! 👋\n\nIt's great to meet another data scientist.  I'm here to help in any way I can. \n\nWhat can I do for you today?  Do you have any questions about a specific project, need help with code, or just want to chat about the latest trends in data science? 😊  \n\nI'm ready when you are! \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 37, 'total_tokens': 121, 'completion_time': 0.152727273, 'prompt_time': 0.002395455, 'queue_time': 0.23011340600000002, 'total_time': 0.155122728}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-ea49c815-45b2-46a7-ab32-f37126250b09-0', usage_metadata={'input_tokens': 37, 'output_tokens': 84, 'total_tokens': 121})

In [55]:
with_message_history = RunnableWithMessageHistory(chain , get_session_history)

config = {
    "configurable" : {"session_id":"chat3"}
}

response = with_message_history.invoke(
    {
        "messages" : [HumanMessage(content="hello my name is Naman and i am a data scientist")]
    },
    config=config
)

response.content

"Hello Naman, it's nice to meet you! As a helpful assistant, I'm here to answer your questions and assist you in any way I can.  \n\nSince you're a data scientist, I imagine you have some interesting projects you're working on. How can I help you today?  \n\nDo you have a specific question about data science, need help with a code snippet, or want to brainstorm ideas for your next project? \n\nLet me know, and I'll do my best to assist!\n"

In [56]:
## adding more complexity
prompt = ChatPromptTemplate.from_messages(
    [
        ("system" , " you are a helpful assistant , answer all the questions to the best of your abilities in this language {language}"), # key 
        MessagesPlaceholder(variable_name="messages"), 
    ]
)

chain = prompt | model 

In [57]:
response = chain.invoke(
    {"messages" : [HumanMessage(content="hello my name is Naman and i am a data scientist")],
    "language":"hindi"}
)

response.content

"नमस्ते नमन, मिलकर अच्छा लगा! मैं आपकी मदद करने के लिए यहाँ हूँ। आपका कोई सवाल है, बेझिझक पूछिए। \n\n(Hello Naman, nice to meet you! I'm here to help you. If you have any questions, feel free to ask.)\n\n"

In [58]:
with_message_history = RunnableWithMessageHistory(
    chain ,
    get_session_history ,
    input_messages_key="messages"
)

config = {"configurable" : {"session_id":"chat4"}}

response = with_message_history.invoke(
    {"messages" : [HumanMessage(content="hello my name is Naman and i am a data scientist")],
    "language":"hindi"},
    config=config
)

response.content

'नमस्ते नमन! 😊 \n\nमुझे बहुत खुशी है कि आपने मेसे मिलकर बातचीत शुरू की है।  एक डेटा साइंटिस्ट होने के नाते आपका काम बहुत ही रोचक है!  \n\nआपके लिए मैं सब कुछ करने के लिए तैयार हूँ। \n\nआप किस बारे में जानना चाहेंगे? 🤔 \n\n'

In [59]:
response = with_message_history.invoke(
    {"messages" : [HumanMessage(content="suggest me some books on data science")],
    "language":"hindi"},
    config=config
)

response.content

'नमन जी, डेटा साइंस के किताबें बहुत सारी हैं,  लेकिन आपकी रुचि और अनुभव के स्तर पर निर्भर करता है। \n\nआपके लिए कुछ सुझाव:\n\n**शुरुआती स्तर के लिए:**\n\n* **"Python for Data Analysis" by Wes McKinney:** पायथन में डेटा एनालिसिस सीखने के लिए एक बेहतरीन किताब।\n* **"Data Science from Scratch" by Joel Grus:** डेटा साइंस के मूल सिद्धांतों को सरल भाषा में समझाया गया है।\n* **"Data Science for Business" by Foster Provost and Tom Fawcett:** डेटा साइंस के व्यावसायिक अनुप्रयोगों पर केंद्रित है।\n\n**मध्यवर्ती स्तर के लिए:**\n\n* **"The Elements of Statistical Learning" by Trevor Hastie, Robert Tibshirani, and Jerome Friedman:**  स्टॅटिस्टिकल मॉडलिंग और मशीन लर्निंग पर एक गहन पुस्तक।\n* **"Pattern Recognition and Machine Learning" by Christopher Bishop:** मशीन लर्निंग के सिद्धांतों और एल्गोरिदम की व्याख्या करता है।\n* **"Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow" by Aurélien Géron:** मशीन लर्निंग के व्यावहारिक अनुप्रयोगों पर ध्यान केंद्रित करता है।\n\n**उन्नत स्तर के लिए:**

# manage the converstaion history and use it to generate the next response

##### trim_message = reduce how many messages we are sending to the model 

In [60]:
from langchain_core.messages import  SystemMessage , trim_messages
from langchain_core.messages import HumanMessage , SystemMessage , AIMessage

trimmer = trim_messages(
    max_tokens=70,
    strategy="last", # last etc
    token_counter = model ,
    include_system = True,
    allow_partial = False,
    start_on = "human"
)


messages = [
    SystemMessage(content = "you are a good assistant"),
    HumanMessage(content = "hello my name is naman and i am a data scientist"),
    AIMessage(content = "hello naman nimble"),  
    HumanMessage(content = "suggest me some books on data science"),
    AIMessage(content = "sure, i can help you with that"),
    HumanMessage(content = "what is the best book on data science"),
    AIMessage(content = "i am not sure about that"),
    HumanMessage(content = "suggest me any best you think"),
    AIMessage(content = "i think you should read the book by xyz"),
]

trimmer.invoke(messages)

[SystemMessage(content='you are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='suggest me some books on data science', additional_kwargs={}, response_metadata={}),
 AIMessage(content='sure, i can help you with that', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the best book on data science', additional_kwargs={}, response_metadata={}),
 AIMessage(content='i am not sure about that', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='suggest me any best you think', additional_kwargs={}, response_metadata={}),
 AIMessage(content='i think you should read the book by xyz', additional_kwargs={}, response_metadata={})]

In [61]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages = itemgetter("messages") | trimmer )  | prompt | model
)

response = chain.invoke(
    {
        "messages":messages + [HumanMessage(content="what was i asking ?")],
        "language":"english"
    }
)

response.content

"You were asking for a suggestion for a good book on data science.  \n\nI apologize for my previous response.  I'm still learning and sometimes get things mixed up. \n\nTo give you a helpful suggestion, could you tell me:\n\n* **What is your current level of experience with data science?** (Beginner, intermediate, advanced)\n* **What are your specific interests within data science?** (Machine learning, statistics, data visualization, etc.)\n* **Do you prefer a more theoretical or practical approach?**\n\n\nKnowing these things will help me recommend a book that's a good fit for you! 😊 \n\n"

In [62]:
# wrapping in message history 

config = {
    "configurable" : {"session_id":"chat5"}
}

response = with_message_history.invoke( 
    {
        "messages":messages + [HumanMessage(content="what was i asking ?")],
        "language":"english"
    },
    config=config
)

response.content

"You were asking me to suggest some good books on data science.  \n\nDo you want me to give you some more specific recommendations?  \n\nFor example, are you looking for a book that covers the fundamentals of data science, or are you more interested in a book that focuses on a specific area like machine learning or deep learning?\n\nWhat's your current level of experience with data science?\n"

In [63]:
response = with_message_history.invoke( 
    {
        "messages":messages + [HumanMessage(content="what is my name ")],
        "language":"english"
    },
    config=config
)

response.content

'Your name is naman.  I remember! 😄 \n\n\nIs there anything else I can help you with today, naman?\n'

In [64]:
response = with_message_history.invoke( 
    {
        "messages":messages + [HumanMessage(content="what was i looking for ? ")],
        "language":"english"
    },
    config=config
)

response.content

"You were looking for book recommendations on data science. \n\nI know it can be tough to choose!  Do you have a particular area of data science you're most interested in? For example, are you more into machine learning, statistical modeling, data visualization, or something else? \n\nKnowing that would help me give you more tailored suggestions. 😊  \n"

In [ ]:
response = with_message_history.invoke( 
    {
        "messages":messages + [HumanMessage(content="what was i looking for ? ")],
        "language":"english"
    },
    config=config
)

response.content

"You were looking for book recommendations on data science.  \n\nIs there a particular area of data science you're most interested in?  For example, are you looking for a book that covers the fundamentals, or something more specialized like machine learning or deep learning?  \n\n Knowing that would help me give you more relevant suggestions! 😊 \n"

# vectorstore and retrievers 

#### langchain_document abstraaction 
##### ->page content  
##### ->metadata ( capture source of info , timestamp , author , etc)

In [66]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="The African Elephant is the largest land animal on Earth. It is known for its large ears, which help regulate body temperature, and its long trunk, used for communication and handling objects. These elephants are primarily found in the savannas and forests of Africa.",
        metadata={"title": "African Elephant", "category": "Mammal", "habitat": "Savanna, Forest", "scientific_name": "Loxodonta africana"}
    ),
    Document(
        page_content="The Bald Eagle is a bird of prey native to North America. It has a striking white head and dark brown body, with a wingspan of up to 7.5 feet. Bald Eagles are powerful hunters and primarily feed on fish. They are also a symbol of the United States.",
        metadata={"title": "Bald Eagle", "category": "Bird", "habitat": "Forests, Near Water", "scientific_name": "Haliaeetus leucocephalus"}
    ),
    Document(
        page_content="The Clownfish is a small, brightly colored fish that lives in coral reefs. It forms a symbiotic relationship with sea anemones, using them for protection against predators. Clownfish are known for their orange bodies with white stripes.",
        metadata={"title": "Clownfish", "category": "Fish", "habitat": "Coral Reefs", "scientific_name": "Amphiprioninae"}
    ),
    Document(
        page_content="The Komodo Dragon is the largest living species of lizard, found in Indonesia. It can grow up to 10 feet long and weighs around 150 pounds. Komodo Dragons are carnivorous and have a strong sense of smell, which helps them hunt prey.",
        metadata={"title": "Komodo Dragon", "category": "Reptile", "habitat": "Islands", "scientific_name": "Varanus komodoensis"}
    ),
    Document(
        page_content="The Blue Whale is the largest animal to have ever lived on Earth. It can reach lengths of over 100 feet and weigh up to 200 tons. Blue Whales feed primarily on tiny shrimp-like animals called krill and are found in all the world's oceans.",
        metadata={"title": "Blue Whale", "category": "Mammal", "habitat": "Oceans", "scientific_name": "Balaenoptera musculus"}
    )
]

In [67]:
documents

[Document(metadata={'title': 'African Elephant', 'category': 'Mammal', 'habitat': 'Savanna, Forest', 'scientific_name': 'Loxodonta africana'}, page_content='The African Elephant is the largest land animal on Earth. It is known for its large ears, which help regulate body temperature, and its long trunk, used for communication and handling objects. These elephants are primarily found in the savannas and forests of Africa.'),
 Document(metadata={'title': 'Bald Eagle', 'category': 'Bird', 'habitat': 'Forests, Near Water', 'scientific_name': 'Haliaeetus leucocephalus'}, page_content='The Bald Eagle is a bird of prey native to North America. It has a striking white head and dark brown body, with a wingspan of up to 7.5 feet. Bald Eagles are powerful hunters and primarily feed on fish. They are also a symbol of the United States.'),
 Document(metadata={'title': 'Clownfish', 'category': 'Fish', 'habitat': 'Coral Reefs', 'scientific_name': 'Amphiprioninae'}, page_content='The Clownfish is a sm

In [69]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(model="Llama3-8b-8192" , groq_api_key=groq_api_key)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000023927E5AA80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023927E5B710>, model_name='Llama3-8b-8192', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
# embeddings 
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name = "ibm-granite/granite-embedding-107m-multilingual") # from hugging face

c:\Users\nex20\Documents\langchain\myenv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nex20\.cache\huggingface\hub\models--ibm-granite--granite-embedding-107m-multilingual. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [81]:
### vectorestore vectors -> embeddings

from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents , embedding = embeddings)
vectorstore

In [86]:
vectorstore.similarity_search("Elephant")

[Document(id='9e214aba-c5f5-49c9-b1db-9f5dbf4a3b2d', metadata={'category': 'Mammal', 'habitat': 'Savanna, Forest', 'scientific_name': 'Loxodonta africana', 'title': 'African Elephant'}, page_content='The African Elephant is the largest land animal on Earth. It is known for its large ears, which help regulate body temperature, and its long trunk, used for communication and handling objects. These elephants are primarily found in the savannas and forests of Africa.'),
 Document(id='93db6316-5a45-460a-947d-2f4a42cad066', metadata={'category': 'Mammal', 'habitat': 'Savanna, Forest', 'scientific_name': 'Loxodonta africana', 'title': 'African Elephant'}, page_content='The African Elephant is the largest land animal on Earth. It is known for its large ears, which help regulate body temperature, and its long trunk, used for communication and handling objects. These elephants are primarily found in the savannas and forests of Africa.'),
 Document(id='8fb19f50-c15f-4702-8141-d9848f7a3d1d', metad

In [87]:
# async query 
await vectorstore.asimilarity_search("African Elephant")

[Document(id='9e214aba-c5f5-49c9-b1db-9f5dbf4a3b2d', metadata={'category': 'Mammal', 'habitat': 'Savanna, Forest', 'scientific_name': 'Loxodonta africana', 'title': 'African Elephant'}, page_content='The African Elephant is the largest land animal on Earth. It is known for its large ears, which help regulate body temperature, and its long trunk, used for communication and handling objects. These elephants are primarily found in the savannas and forests of Africa.'),
 Document(id='93db6316-5a45-460a-947d-2f4a42cad066', metadata={'category': 'Mammal', 'habitat': 'Savanna, Forest', 'scientific_name': 'Loxodonta africana', 'title': 'African Elephant'}, page_content='The African Elephant is the largest land animal on Earth. It is known for its large ears, which help regulate body temperature, and its long trunk, used for communication and handling objects. These elephants are primarily found in the savannas and forests of Africa.'),
 Document(id='8fb19f50-c15f-4702-8141-d9848f7a3d1d', metad

In [89]:
vectorstore.similarity_search_with_score("reptile")

[(Document(id='8fb19f50-c15f-4702-8141-d9848f7a3d1d', metadata={'category': 'Reptile', 'habitat': 'Islands', 'scientific_name': 'Varanus komodoensis', 'title': 'Komodo Dragon'}, page_content='The Komodo Dragon is the largest living species of lizard, found in Indonesia. It can grow up to 10 feet long and weighs around 150 pounds. Komodo Dragons are carnivorous and have a strong sense of smell, which helps them hunt prey.'),
  0.7592587471008301),
 (Document(id='d6d595b6-31ac-474a-ae0b-e68c2734674f', metadata={'category': 'Reptile', 'habitat': 'Islands', 'scientific_name': 'Varanus komodoensis', 'title': 'Komodo Dragon'}, page_content='The Komodo Dragon is the largest living species of lizard, found in Indonesia. It can grow up to 10 feet long and weighs around 150 pounds. Komodo Dragons are carnivorous and have a strong sense of smell, which helps them hunt prey.'),
  0.7592587471008301),
 (Document(id='9e214aba-c5f5-49c9-b1db-9f5dbf4a3b2d', metadata={'category': 'Mammal', 'habitat': '

## retrievals of documents 
#### -> do not sublass runnable cannot interact with lcel 
#### -> runnables implement standard set of methods ( synchronous and asysnchronous invoke and batch operations)


In [ ]:
from typing import List  

from langchain_core.documents import Document
from langchain_core.runnables import  RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k = 1) # bind gives most similar document
retriever.batch(["lizard","elephant"])

[[Document(id='8fb19f50-c15f-4702-8141-d9848f7a3d1d', metadata={'category': 'Reptile', 'habitat': 'Islands', 'scientific_name': 'Varanus komodoensis', 'title': 'Komodo Dragon'}, page_content='The Komodo Dragon is the largest living species of lizard, found in Indonesia. It can grow up to 10 feet long and weighs around 150 pounds. Komodo Dragons are carnivorous and have a strong sense of smell, which helps them hunt prey.')],
 [Document(id='9e214aba-c5f5-49c9-b1db-9f5dbf4a3b2d', metadata={'category': 'Mammal', 'habitat': 'Savanna, Forest', 'scientific_name': 'Loxodonta africana', 'title': 'African Elephant'}, page_content='The African Elephant is the largest land animal on Earth. It is known for its large ears, which help regulate body temperature, and its long trunk, used for communication and handling objects. These elephants are primarily found in the savannas and forests of Africa.')]]

#### second method : as retrieval of documents
#### -> use the retriever to get the documents
#### -> better way
#### -> retrievals are runnables (imp)

In [92]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)

retriever.batch(["lizard","elephant"])

[[Document(id='8fb19f50-c15f-4702-8141-d9848f7a3d1d', metadata={'category': 'Reptile', 'habitat': 'Islands', 'scientific_name': 'Varanus komodoensis', 'title': 'Komodo Dragon'}, page_content='The Komodo Dragon is the largest living species of lizard, found in Indonesia. It can grow up to 10 feet long and weighs around 150 pounds. Komodo Dragons are carnivorous and have a strong sense of smell, which helps them hunt prey.')],
 [Document(id='9e214aba-c5f5-49c9-b1db-9f5dbf4a3b2d', metadata={'category': 'Mammal', 'habitat': 'Savanna, Forest', 'scientific_name': 'Loxodonta africana', 'title': 'African Elephant'}, page_content='The African Elephant is the largest land animal on Earth. It is known for its large ears, which help regulate body temperature, and its long trunk, used for communication and handling objects. These elephants are primarily found in the savannas and forests of Africa.')]]

# RAG

In [93]:
from langchain_core.prompts import ChatPromptTemplate 
from langchain_core.runnables   import RunnablePassthrough

messages = """
            answer the questions with provided context only 
            {question}
            
            context : {context}
"""

In [96]:
prompt = ChatPromptTemplate.from_messages(
    ["human" , messages]
)

rag_chain = {"context":retriever,"question":RunnablePassthrough()}| prompt | llm

response = rag_chain.invoke(
    "tell me about elephants"
)

print(response.content)

The African Elephant, being the largest land animal on Earth, is known for its distinctive features such as its large ears, which help regulate its body temperature, and its long trunk, used for communication and handling objects.
